# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [19]:
# Signal 1: Search visibility volume
# March 2026 is used as the decision-time development window.

daily_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

signal1 = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print("Rows:", len(signal1))
print(signal1["impressions"].describe())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
count    176738.000000
mean       1587.986675
std        5431.337724
min           1.000000
25%          20.000000
50%         173.000000
75%        1039.000000
max      617124.000000
Name: impressions, dtype: float64


In [20]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

daily_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

print("DuckDB + Hugging Face connection ready!")

Token loaded: True
DuckDB + Hugging Face connection ready!


In [21]:
signal1 = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions
    FROM read_parquet(
        '{daily_path}',
        hive_partitioning=true
    )
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print("Signal 1 rows:", len(signal1))
print()
print(signal1["impressions"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 rows: 176738

count    176738.000000
mean       1587.986675
std        5431.337724
min           1.000000
25%          20.000000
50%         173.000000
75%        1039.000000
max      617124.000000
Name: impressions, dtype: float64


### Signal 1 verdict: CONFIRMED

The impression buckets show clear separation in observed search exposure: the median impressions increase from 4 in Q1 to 3,012 in Q4 High. This confirms that impressions are a useful visibility-volume signal for identifying content with meaningful search exposure. The signal is used only as a prioritization input, not as evidence that higher impressions cause better performance.

In [22]:
signal1["impression_bucket"] = pd.qcut(
    signal1["impressions"],
    q=4,
    labels=["Q1 Low", "Q2", "Q3", "Q4 High"],
    duplicates="drop"
)

bucket1 = (
    signal1
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("impressions", "size"),
        median_impressions=("impressions", "median"),
        mean_impressions=("impressions", "mean")
    )
    .reset_index()
)

print(bucket1.to_string(index=False))

impression_bucket     n  median_impressions  mean_impressions
           Q1 Low 44983                 4.0          6.230998
               Q2 43409                67.0         76.723491
               Q3 44186               419.0        478.295908
          Q4 High 44160              3012.0       5795.127423


In [23]:
import pandas as pd

In [24]:
signal2 = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr
    FROM read_parquet(
        '{daily_path}',
        hive_partitioning=true
    )
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print("Signal 2 rows:", len(signal2))
print()
print(signal2["ctr"].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2 rows: 176738

count    176738.000000
mean          0.004594
std           0.037760
min           0.000000
25%           0.000000
50%           0.000000
75%           0.002158
max           1.000000
Name: ctr, dtype: float64


In [25]:
signal2["ctr_bucket"] = pd.cut(
    signal2["ctr"],
    bins=[-0.000001, 0, 0.001, 0.005, 1.000001],
    labels=["0% CTR", "0-0.1%", "0.1-0.5%", "0.5%+"],
    include_lowest=True
)

bucket2 = (
    signal2
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print(bucket2.to_string(index=False))

ctr_bucket      n  median_ctr  mean_ctr
    0% CTR 107901    0.000000  0.000000
    0-0.1%   9657    0.000647  0.000622
  0.1-0.5%  37012    0.002488  0.002657
     0.5%+  22168    0.009050  0.031919


### Signal 2 verdict: CONFIRMED

The CTR buckets show clear separation in observed click-through performance. The median CTR rises from 0% in the 0% CTR bucket to 0.905% in the 0.5%+ bucket. This confirms that CTR is a useful signal for identifying content that receives search exposure but captures relatively few clicks. Because low CTR can also occur on very low-volume content, CTR should be interpreted together with search visibility rather than used alone.

In [26]:
import pandas as pd

starter_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

refresh_df = pd.read_csv(starter_url)

print("Rows:", len(refresh_df))
print("Columns:")
print(refresh_df.columns.tolist())

refresh_df.head()

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [27]:
# Signal 3: Content staleness / freshness
# This is the refresh-flag-linked signal.

print("Staleness summary:")
print(refresh_df["days_since_last_update"].describe())

print("\nFreshness tiers:")
print(
    refresh_df["freshness_tier"]
    .value_counts(dropna=False)
    .to_string()
)

Staleness summary:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Freshness tiers:
freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174


In [28]:
refresh_df["staleness_bucket"] = pd.cut(
    refresh_df["days_since_last_update"],
    bins=[0, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"],
    include_lowest=True
)

staleness_table = (
    refresh_df
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("days_since_last_update", "size"),
        median_days=("days_since_last_update", "median"),
        mean_days=("days_since_last_update", "mean")
    )
    .reset_index()
)

print(staleness_table.to_string(index=False))

staleness_bucket     n  median_days  mean_days
       0-30 days 20480         20.0  18.540918
      31-90 days   175         41.0  53.628571
     91-180 days  9171        104.0 104.106204
       181+ days   174        211.0 224.643678


### Staleness signal verdict: CONFIRMED

The staleness buckets show clear separation in time since last update, with median staleness increasing from 20 days in the 0–30 day bucket to 211 days in the 181+ day bucket. This confirms that days since last update provides a direct, interpretable freshness signal for a refresh-review rule. The signal identifies age of the last update, but does not by itself prove that refreshing a page will improve search performance.

### Final baseline rule

I will prioritize content for refresh review when it has meaningful search visibility, relatively low CTR, and has not been updated recently.

The baseline score will combine:
- search impressions as the visibility signal,
- CTR as the click-capture signal,
- days since last update as the refresh/staleness signal.

Higher impressions increase priority, lower CTR increases priority, and greater days since last update increases priority.

### Reason code

`LOW_CTR_STALE_VISIBLE`

### Action label

`REVIEW_REFRESH`

This is a directional decision-support rule based on observed signals. It does not claim that refreshing content will cause improved search performance.

In [29]:
# Inspect the decision-time fields we will use for the baseline score

queue_df = refresh_df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "days_since_last_update",
        "freshness_tier"
    ]
].copy()

print("Rows:", len(queue_df))
print(queue_df.head(10).to_string(index=False))

Rows: 30000
          content_id         client_id  impressions_90d  clicks_90d  ctr  days_since_last_update freshness_tier
content_304f48230142 client_f369cb89fc             3803          29 0.76                      20           0-30
content_a1fb4e703a9e client_4e07408562            15320           7 0.05                      25           0-30
content_9aa793d4d895 client_7f2253d7e2            12581          11 0.09                      20           0-30
content_331d6c4de07b client_19581e27de            11751          58 0.49                      22           0-30
content_d99b7a2d90ca client_3fdba35f04            19140          24 0.13                      14           0-30
content_d4084a4bc775 client_f369cb89fc             3970           1 0.03                      20           0-30
content_9a34b442b552 client_8722616204               20           0 0.00                      20           0-30
content_a63219c6e95a client_19581e27de             1724           1 0.06                    

In [30]:
# Build the baseline score using only decision-time signals.

queue_df = refresh_df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "days_since_last_update",
        "freshness_tier"
    ]
].copy()

# Remove rows where the required decision-time signals are missing.
queue_df = queue_df.dropna(
    subset=["impressions_90d", "ctr", "days_since_last_update"]
).copy()

# Convert each signal into a percentile rank from 0 to 1.
# Higher impressions = more visibility.
queue_df["visibility_score"] = (
    queue_df["impressions_90d"].rank(pct=True)
)

# Lower CTR = higher refresh opportunity.
queue_df["low_ctr_score"] = (
    1 - queue_df["ctr"].rank(pct=True)
)

# Older content = higher refresh opportunity.
queue_df["staleness_score"] = (
    queue_df["days_since_last_update"].rank(pct=True)
)

# One baseline score: 40% visibility + 35% low CTR + 25% staleness.
queue_df["baseline_score"] = (
    0.40 * queue_df["visibility_score"]
    + 0.35 * queue_df["low_ctr_score"]
    + 0.25 * queue_df["staleness_score"]
) * 100

queue_df["reason_code"] = "LOW_CTR_STALE_VISIBLE"
queue_df["action"] = "REVIEW_REFRESH"

# Highest score gets rank 1.
queue_df = queue_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue_df["rank"] = queue_df.index + 1

print("Ranked rows:", len(queue_df))
print()
print(
    queue_df[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(10).to_string(index=False)
)



# Write the ranked baseline queue.
# The CSV is regenerated from the notebook and should stay out of git.

output_path = "/content/baseline_action_score.csv"

queue_df[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "days_since_last_update"
    ]
].to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)

print()
print("Rows written:", len(queue_df))

Ranked rows: 30000

 rank           content_id  impressions_90d  ctr  days_since_last_update  baseline_score           reason_code         action
    1 content_c8e9d6ab9013           208678  0.0                     104       88.335083 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    2 content_fb4bf6555c79            84093  0.0                     104       88.067083 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    3 content_6e28a04c07a8            41226  0.0                     104       87.473750 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    4 content_bc18d49d8f6b            32491  0.0                     104       87.097750 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    5 content_b21385c39124            30962  0.0                     104       87.000417 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    6 content_75175d878762            25748  0.0                     104       86.600417 LOW_CTR_STALE_VISIBLE REVIEW_REFRESH
    7 content_095661034f9b            23513  0.0                     104       86.421750 LOW_CTR_S

In [31]:
from pathlib import Path

output_path = Path("/content/work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue_df[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "ctr",
        "days_since_last_update"
    ]
].to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)
print("Rows written:", len(queue_df))

CSV written successfully:
/content/work/outputs/baseline_action_score.csv
Rows written: 30000


In [32]:
top20 = queue_df.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda r: (
        f"High visibility ({int(r['impressions_90d']):,} impressions), "
        f"low CTR ({r['ctr']:.2f}), and stale update signal "
        f"({int(r['days_since_last_update'])} days)."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "Low CTR may be explained by search intent, SERP context, "
    "or data/tracking issues rather than a refresh opportunity."
)

print(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ].to_string(index=False)
)

 rank           content_id         action           reason_code                                                                            confidence_note                                                                                            what_would_make_it_wrong
    1 content_c8e9d6ab9013 REVIEW_REFRESH LOW_CTR_STALE_VISIBLE High visibility (208,678 impressions), low CTR (0.00), and stale update signal (104 days). Low CTR may be explained by search intent, SERP context, or data/tracking issues rather than a refresh opportunity.
    2 content_fb4bf6555c79 REVIEW_REFRESH LOW_CTR_STALE_VISIBLE  High visibility (84,093 impressions), low CTR (0.00), and stale update signal (104 days). Low CTR may be explained by search intent, SERP context, or data/tracking issues rather than a refresh opportunity.
    3 content_6e28a04c07a8 REVIEW_REFRESH LOW_CTR_STALE_VISIBLE  High visibility (41,226 impressions), low CTR (0.00), and stale update signal (104 days). Low CTR may be explained by sear

In [33]:
# Identify potentially weak picks among the top 20
median_impressions = queue_df["impressions_90d"].median()

weak_picks = queue_df.head(20).copy()
weak_picks = weak_picks[
    weak_picks["impressions_90d"] < median_impressions
]

print("Median impressions in ranked queue:", median_impressions)
print("\nPotential weak picks among Top-20:")

if len(weak_picks) == 0:
    print("None — all Top-20 picks have impressions above the queue median.")
else:
    print(
        weak_picks[
            [
                "rank",
                "content_id",
                "impressions_90d",
                "ctr",
                "days_since_last_update",
                "baseline_score"
            ]
        ].to_string(index=False)
    )

print("\n--- Leakage Check ---")

used_features = [
    "impressions_90d",
    "ctr",
    "days_since_last_update"
]

print("Features used in baseline:")
for feature in used_features:
    print("-", feature)

future_like_features = [
    col for col in refresh_df.columns
    if any(word in col.lower() for word in ["future", "label", "target", "outcome"])
]

print("\nFuture/label-like columns found in starter data:")
print(future_like_features if future_like_features else "None")

print("\nLeakage verdict: PASS")
print("The baseline score uses only observed visibility, CTR, and update-recency signals.")
print("No future outcome or label-derived feature is used in the scoring formula.")

Median impressions in ranked queue: 731.0

Potential weak picks among Top-20:
None — all Top-20 picks have impressions above the queue median.

--- Leakage Check ---
Features used in baseline:
- impressions_90d
- ctr
- days_since_last_update

Future/label-like columns found in starter data:
None

Leakage verdict: PASS
The baseline score uses only observed visibility, CTR, and update-recency signals.
No future outcome or label-derived feature is used in the scoring formula.


### Self-check

* **Two signals checked:** Yes — search visibility volume and CTR were checked with bucket tables and printed sample counts.
* **Flag-linked signal checked:** Yes — staleness was checked using the refresh/freshness signal from the starter data.
* **One baseline rule:** Yes — visibility, low CTR, and staleness are combined into one baseline score.
* **One reason code:** Yes — `LOW_CTR_STALE_VISIBLE`.
* **One action label:** Yes — `REVIEW_REFRESH`.
* **Ranked queue created:** Yes — 30,000 rows were ranked and written to `work/outputs/baseline_action_score.csv`.
* **Top-20 reviewed:** Yes — each pick includes an action, reason code, confidence note, and what could make it wrong.
* **Weak picks checked:** Yes — no Top-20 pick fell below the queue median for impressions.
* **Leakage check:** PASS — no future outcome or label-derived feature was used.
* **Interpretation:** This is a directional prioritization rule for human review, not a causal claim that refreshing content will improve search performance.
